In [1]:
from pathlib import Path
import pandas as pd

import pickle
DATA_DIR = Path("/root/capsule/data")
SCRATCH_DIR = Path("/root/capsule/scratch")
MATCH_DIR = SCRATCH_DIR / "ophys_xenium_match_tables"
MATCH_DIR.mkdir(exist_ok=True)

load data information

In [ ]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/xenium_data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

mouse_ids: [778174, 786297, 797371]


# unifed table connecting ophys sessions and xenium

In [ ]:
for mouse_id in mouse_ids:
    ophys_session_names = {asset_name: data_info[mouse_id]['ophys'][asset_name]['raw'] for asset_name in data_info[mouse_id]['ophys'] if data_info[mouse_id]['ophys'][asset_name]['session_type'] == 'STAGE_1'}
    # ophys_session_names = {asset_name: data_info[mouse_id]['ophys'][asset_name]['raw'] for asset_name in data_info[mouse_id]['ophys']}

    ophys_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['ophys_zstack']
    ophys_zstack_path = ophys_zstack_path / list(ophys_zstack_path.glob("*"))[0]
    planes = [f"VISp_{i}" for i in range(8)]
    ophys_zstack_match = []
    for plane in planes:
        for session, session_name in ophys_session_names.items():
            cell_matching_path = ophys_zstack_path / session_name / 'cell_matching' / f"{plane}_to_zstack_match.csv"
            df_match = pd.read_csv(cell_matching_path)[['mask_id_fov','mask_id_cz','plane','mouse_id','x_cz','y_cz','z_cz']]
            df_match.rename(columns={'mask_id_fov': f'cell_id - {session}'}, inplace=True)
            if session == list(ophys_session_names.keys())[0]:
                df_plane = df_match.copy()
            else:
                df_plane = pd.merge(df_plane, df_match, on=['mask_id_cz','plane','mouse_id','x_cz','y_cz','z_cz'], how='left')
        # After merging all sessions, drop rows with any NaN cell_id and cast to int
        for session in ophys_session_names:
            df_plane.dropna(subset=[f'cell_id - {session}'], inplace=True)
            df_plane[f'cell_id - {session}'] = df_plane[f'cell_id - {session}'].astype(int)
        ophys_zstack_match.append(df_plane)

    ophys_zstack_match = pd.concat(ophys_zstack_match,axis= 0)
    ophys_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack', 'x_cz': 'x_loc', 'y_cz': 'y_loc', 'z_cz': 'z_loc'}, inplace=True)
    ophys_zstack_match = ophys_zstack_match[['mouse_id', 'plane', 'cell_id - zstack', 'x_loc', 'y_loc', 'z_loc']+[ f'cell_id - {session}' for session in ophys_session_names]]

    xenium_zstack_match = []
    xenium_zstack_path = DATA_DIR / data_info[mouse_id]['coregistration']['xenium_zstack'] / "cell_matching"
    cell_matching_files =   list(xenium_zstack_path.glob("*.csv"))
    for file in cell_matching_files:
        df = pd.read_csv(file)
        df['xenium_section'] = int(file.stem.split('_')[1])
        df['mouse_id'] = mouse_id
        mask_cz_counts = df.groupby('mask_id_cz')['mask_id_xenium'].nunique()
        df = df[df['iou'] > 0.4]
        duplicate_mask_cz = mask_cz_counts[mask_cz_counts > 1].index
        df = df[~df['mask_id_cz'].isin(duplicate_mask_cz)]
        xenium_zstack_match.append(df)
    
    xenium_zstack_match = pd.concat(xenium_zstack_match, ignore_index=True)
    xenium_zstack_match = xenium_zstack_match.sort_values(['iou']).drop_duplicates(['mouse_id', 'mask_id_cz'], keep='first')

    xenium_zstack_match.rename(columns={'mask_id_cz': 'cell_id - zstack', 'mask_id_xenium': 'cell_id - xenium'}, inplace=True)
    xenium_zstack_match.sort_values(['mouse_id', 'xenium_section', 'cell_id - zstack'], inplace=True)
    xenium_zstack_match = xenium_zstack_match[['mouse_id', 'xenium_section', 'cell_id - zstack', 'cell_id - xenium']]
    xenium_zstack_match

    ophys_xenium_match = xenium_zstack_match.merge(ophys_zstack_match, on=['mouse_id', 'cell_id - zstack'], how='inner')
    ophys_xenium_match = ophys_xenium_match.drop_duplicates(subset=['mouse_id', 'xenium_section','cell_id - xenium'])
    ophys_xenium_match.insert(0, 'unique_id', ophys_xenium_match.apply(lambda row:( f"{row['mouse_id']}_{row['xenium_section']}_{row['cell_id - zstack']}"), axis=1))
    ophys_xenium_match.to_csv(MATCH_DIR / f'{mouse_id}x_ophys_transcriptomics_match.csv', index = False)
    print(mouse_id, ophys_xenium_match.shape)


778174 (821, 12)
786297 (1425, 12)
797371 (1302, 12)


In [4]:
ophys_xenium_match

,unique_id,mouse_id,xenium_section,cell_id - zstack,cell_id - xenium,plane,x_loc,y_loc,z_loc,cell_id - session_1,cell_id - session_2,cell_id - session_3
0,797371_1_705,797371,1,705,34963,VISp_4,336.312614,431.243601,121.442261,129,145,157
1,797371_1_737,797371,1,737,15836,VISp_4,429.118122,255.111372,121.406456,24,34,43
2,797371_1_738,797371,1,738,18449,VISp_4,250.906733,285.249363,122.451192,48,54,65
3,797371_1_865,797371,1,865,15829,VISp_4,256.615658,226.113818,124.777059,4,12,24
4,797371_2_643,797371,2,643,4034,VISp_4,570.830363,602.354570,121.510049,223,236,252
...,...,...,...,...,...,...,...,...,...,...,...,...
1468,797371_22_14972,797371,22,14972,10704,VISp_7,565.533513,571.595954,376.737694,265,271,290
1469,797371_22_14977,797371,22,14977,7985,VISp_7,516.286434,634.210836,370.939941,298,306,327
1470,797371_22_15069,797371,22,15069,10679,VISp_7,629.980086,488.764378,379.938797,223,214,234
1471,797371_22_15076,797371,22,15076,7991,VISp_7,496.720414,602.781479,376.309866,279,290,307
